In [4]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans, SpectralClustering
from sklearn.impute import SimpleImputer
from sklearn.mixture import GaussianMixture
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler


RANDOM_STATE = 42


def find_first_existing_path(paths):
    for path in paths:
        if path.exists():
            return path.resolve()
    tried = "\n".join(str(p) for p in paths)
    raise FileNotFoundError(f"Could not find input CSV. Tried:\n{tried}")


def impute_median(frame):
    imputer = SimpleImputer(strategy="median")
    values = imputer.fit_transform(frame)
    return pd.DataFrame(values, columns=frame.columns, index=frame.index)


def zscore_frame(frame):
    std = frame.std(ddof=0).replace(0, 1)
    return (frame - frame.mean()) / std


candidate_input_paths = [
    Path("features.csv"),
    Path("modelling/features.csv"),
    Path.cwd() / "features.csv",
    Path.cwd() / "modelling" / "features.csv",
]
input_csv_path = find_first_existing_path(candidate_input_paths)
df_raw = pd.read_csv(input_csv_path)

id_candidates = ["user_id", "userid", "userId", "id"]
id_col = next((col for col in id_candidates if col in df_raw.columns), df_raw.columns[0])
results = pd.DataFrame({id_col: df_raw[id_col]})

# 1) Platform Engagement: Disengaged / Moderate / Highly Engaged
platform_cols = ["active_days", "total_active_learning_time_seconds"]
platform_x = impute_median(df_raw[platform_cols].clip(lower=0))
platform_x = np.log1p(platform_x)
platform_model = KMeans(n_clusters=3, random_state=RANDOM_STATE, n_init="auto")
platform_cluster = platform_model.fit_predict(platform_x)
platform_centers = pd.DataFrame(platform_model.cluster_centers_, columns=platform_cols)
platform_score = zscore_frame(platform_centers).sum(axis=1)
platform_order = platform_score.sort_values().index.tolist()
platform_map = {
    platform_order[0]: "Disengaged",
    platform_order[1]: "Moderate",
    platform_order[2]: "Highly Engaged",
}
results["platform_engagement_label"] = pd.Series(platform_cluster, index=df_raw.index).map(platform_map)

# 2) Activity Regularity: Crammer / Moderate / Consistent Learner
activity_cols = ["login_gap_days_mean", "login_gap_days_var"]
activity_x = impute_median(df_raw[activity_cols].clip(lower=0))
activity_x["login_gap_days_var"] = np.log1p(activity_x["login_gap_days_var"])
activity_model = KMeans(n_clusters=3, random_state=RANDOM_STATE, n_init="auto")
activity_cluster = activity_model.fit_predict(activity_x)
activity_centers = pd.DataFrame(activity_model.cluster_centers_, columns=activity_cols)
activity_irregularity = zscore_frame(activity_centers).sum(axis=1)
activity_order = activity_irregularity.sort_values().index.tolist()
activity_map = {
    activity_order[0]: "Consistent Learner",
    activity_order[1]: "Moderate",
    activity_order[2]: "Crammer",
}
results["activity_regularity_label"] = pd.Series(activity_cluster, index=df_raw.index).map(activity_map)

# 3) Learning Resource Interaction Depth: Surface / Moderate / Deep Learner
content_cols = [
    "avg_video_completion_ratio",
    "videos_interacted",
    "pause_per_video",
    "replay_per_video",
    "avg_scroll_depth_ratio",
]
content_x = impute_median(df_raw[content_cols].clip(lower=0))
content_scaled = StandardScaler().fit_transform(content_x)
if len(content_x) > 2:
    n_neighbors = min(10, len(content_x) - 1)
    distances, _ = NearestNeighbors(n_neighbors=n_neighbors).fit(content_scaled).kneighbors(content_scaled)
    kth_dist = distances[:, -1]
    positive_dist = kth_dist[kth_dist > 0]
    sigma = float(np.median(positive_dist)) if positive_dist.size > 0 else 1.0
else:
    sigma = 1.0
gamma = 1.0 / (2.0 * sigma * sigma) if sigma > 0 else 1.0
content_model = SpectralClustering(
    n_clusters=3,
    affinity="rbf",
    gamma=gamma,
    assign_labels="kmeans",
    random_state=RANDOM_STATE,
)
content_cluster = content_model.fit_predict(content_scaled)
content_centers = content_x.assign(cluster=content_cluster).groupby("cluster")[content_cols].mean()
content_depth_score = zscore_frame(content_centers).mean(axis=1)
content_order = content_depth_score.sort_values().index.tolist()
content_map = {
    content_order[0]: "Surface Learner",
    content_order[1]: "Moderate Learner",
    content_order[2]: "Deep Learner",
}
results["learning_resource_interaction_depth_label"] = pd.Series(content_cluster, index=df_raw.index).map(content_map)

# 4) Help-Seeking: Independent / Effective Seeker / Passive Seeker
help_cols = ["avg_hints_per_question", "hint_to_accuracy_ratio"]
help_x = impute_median(df_raw[help_cols].clip(lower=0))
hint_cap = help_x["avg_hints_per_question"].quantile(0.99)
help_x["avg_hints_per_question"] = help_x["avg_hints_per_question"].clip(upper=hint_cap)
help_model = KMeans(n_clusters=3, random_state=RANDOM_STATE, n_init="auto")
help_cluster = help_model.fit_predict(help_x)
help_centers = pd.DataFrame(help_model.cluster_centers_, columns=help_cols)
help_z = zscore_frame(help_centers)
help_passive_signal = help_z["avg_hints_per_question"] + help_z["hint_to_accuracy_ratio"]
passive_cluster = int(help_passive_signal.idxmax())
independent_cluster = int(help_passive_signal.idxmin())
effective_cluster = [c for c in help_centers.index if c not in {passive_cluster, independent_cluster}][0]
help_map = {
    independent_cluster: "Independent",
    effective_cluster: "Effective Seeker",
    passive_cluster: "Passive Seeker",
}
results["help_seeking_label"] = pd.Series(help_cluster, index=df_raw.index).map(help_map)

# 5) AI Interaction Behavior: Non-User / Answer-Seeker / Concept Explorer
ai_cols = ["chatbot_user_messages_per_thread", "chatbot_prompt_effort_chars"]
ai_x = df_raw[ai_cols].fillna(0).clip(lower=0)
non_user_mask = ai_x.sum(axis=1) <= 0
ai_labels = pd.Series("Non-User", index=df_raw.index, dtype="object")
if (~non_user_mask).sum() >= 2:
    ai_active = np.log1p(ai_x.loc[~non_user_mask])
    ai_model = KMeans(n_clusters=2, random_state=RANDOM_STATE, n_init="auto")
    ai_cluster = pd.Series(ai_model.fit_predict(ai_active), index=ai_active.index)
    ai_centers = pd.DataFrame(ai_model.cluster_centers_, columns=ai_cols)
    ai_score = zscore_frame(ai_centers).sum(axis=1)
    concept_cluster = int(ai_score.idxmax())
    answer_cluster = [c for c in ai_centers.index if c != concept_cluster][0]
    ai_map = {
        answer_cluster: "Answer-Seeker",
        concept_cluster: "Concept Explorer",
    }
    ai_labels.loc[~non_user_mask] = ai_cluster.map(ai_map)
results["ai_interaction_behavior_label"] = ai_labels

# 6) Reflective Behavior: Non-Adaptive / Gradually Adaptive / Highly Adaptive
recovery = df_raw["error_recovery_rate"]
reflective_label = pd.Series(index=df_raw.index, dtype="object")
recovery_valid = recovery.dropna()
if recovery_valid.empty:
    reflective_label[:] = "Non-Adaptive"
else:
    ranked_valid = recovery_valid.rank(method="first")
    reflective_bins = pd.qcut(
        ranked_valid,
        q=3,
        labels=["Non-Adaptive", "Gradually Adaptive", "Highly Adaptive"],
    )
    reflective_label.loc[recovery_valid.index] = reflective_bins.astype(str)
    reflective_label = reflective_label.fillna("Non-Adaptive")
results["reflective_behavior_label"] = reflective_label

# 7) High Achievement: Struggling / Inconsistent / Proficient / Mastery
achievement_cols = ["average_score_ratio", "score_variance"]
achievement_x = impute_median(df_raw[achievement_cols])
achievement_x["average_score_ratio"] = achievement_x["average_score_ratio"].clip(lower=0, upper=1)
achievement_x["score_variance"] = achievement_x["score_variance"].clip(lower=0)
achievement_model = GaussianMixture(n_components=4, random_state=RANDOM_STATE)
achievement_cluster = achievement_model.fit_predict(achievement_x)
achievement_centers = achievement_x.assign(cluster=achievement_cluster).groupby("cluster")[achievement_cols].mean()
achievement_z = zscore_frame(achievement_centers)
mastery_cluster = int((achievement_z["average_score_ratio"] - achievement_z["score_variance"]).idxmax())
struggling_cluster = int(achievement_z["average_score_ratio"].idxmin())
remaining_clusters = [c for c in achievement_centers.index if c not in {mastery_cluster, struggling_cluster}]
inconsistent_cluster = int(achievement_z.loc[remaining_clusters, "score_variance"].idxmax())
proficient_cluster = [c for c in remaining_clusters if c != inconsistent_cluster][0]
achievement_map = {
    struggling_cluster: "Struggling",
    inconsistent_cluster: "Inconsistent",
    proficient_cluster: "Proficient",
    mastery_cluster: "Mastery",
}
results["high_achievement_label"] = pd.Series(achievement_cluster, index=df_raw.index).map(achievement_map)

# 8) High Growth: Declining / Stagnant / Diverging / Thriving
eps = 1e-4
score_trend = df_raw["score_trend"].fillna(0)
engagement_trend = df_raw["engagement_trend"].fillna(0)
results["high_growth_label"] = np.select(
    [
        (score_trend > eps) & (engagement_trend > eps),
        (score_trend < -eps) & (engagement_trend < -eps),
        ((score_trend > eps) & (engagement_trend < -eps))
        | ((score_trend < -eps) & (engagement_trend > eps)),
    ],
    ["Thriving", "Declining", "Diverging"],
    default="Stagnant",
)

output_csv_path = input_csv_path.parent / "user_category_results.csv"
results.to_csv(output_csv_path, index=False)

print(f"Input: {input_csv_path}")
print(f"Output: {output_csv_path}")
print(f"Rows: {len(results)}")
results.head()

Input: /home/dicksonyu1230/exchange/MLBD/CS_421_MLBD/modelling/features.csv
Output: /home/dicksonyu1230/exchange/MLBD/CS_421_MLBD/modelling/user_category_results.csv
Rows: 799


,user_id,platform_engagement_label,activity_regularity_label,learning_resource_interaction_depth_label,help_seeking_label,ai_interaction_behavior_label,reflective_behavior_label,high_achievement_label,high_growth_label
0,127,Moderate,Moderate,Surface Learner,Passive Seeker,Answer-Seeker,Highly Adaptive,Mastery,Diverging
1,525,Moderate,Crammer,Surface Learner,Independent,Answer-Seeker,Non-Adaptive,Inconsistent,Diverging
2,564,Disengaged,Crammer,Surface Learner,Independent,Concept Explorer,Non-Adaptive,Struggling,Diverging
3,568,Moderate,Moderate,Surface Learner,Independent,Concept Explorer,Gradually Adaptive,Proficient,Diverging
4,569,Moderate,Moderate,Surface Learner,Independent,Concept Explorer,Highly Adaptive,Proficient,Declining
